In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt

In [71]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)

display(train_df.head())
display(test_df.head())

Training shape: (9864, 19)
Testing shape: (2466, 18)


,Session_ID,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,112163,0,0.00,0,0.0,3,44.500000,0.066667,0.133333,0.000000,0.0,Dec,2,2,9.0,3.0,Returning_Visitor,False,False
1,107490,0,0.00,0,0.0,12,460.200000,0.061111,0.111111,0.000000,0.0,Jul,2,2,6.0,4.0,Returning_Visitor,False,False
2,106273,4,48.80,0,0.0,11,344.800000,0.015385,0.054396,0.000000,0.0,Oct,3,2,1.0,4.0,Returning_Visitor,True,False
3,110651,0,0.00,0,0.0,23,517.035714,0.000000,0.009524,23.300007,0.0,Dec,4,2,8.0,2.0,New_Visitor,False,True
4,101259,7,110.25,0,0.0,20,266.583333,0.011111,0.039753,0.000000,0.0,Mar,2,2,1.0,2.0,Returning_Visitor,False,False


,Session_ID,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend
0,106094,0,0.000000,0,0.000000,8,286.700000,0.050000,0.075000,0.000000,0.0,Jul,3,2,1.0,1.0,Returning_Visitor,False
1,111845,2,10.000000,1,19.000000,24,845.833333,0.004000,0.022000,0.000000,0.0,Dec,1,1,4.0,3.0,Returning_Visitor,True
2,106794,10,215.244507,2,245.171429,101,3558.576942,0.008636,0.012041,30.872272,0.0,Sep,3,2,1.0,2.0,Returning_Visitor,True
3,103444,2,81.000000,0,0.000000,18,631.000000,0.010000,0.029000,0.000000,0.0,May,2,2,1.0,1.0,Returning_Visitor,False
4,106833,0,0.000000,0,0.000000,45,1387.267857,0.036508,0.059221,0.000000,0.0,Jul,3,2,9.0,13.0,Returning_Visitor,False


In [72]:
original_df = train_df.copy()

print("Original training shape:", original_df.shape)
print("Original missing values:", original_df.isnull().sum().sum())

Original training shape: (9864, 19)
Original missing values: 2957


In [73]:
print("Training data information:")
train_df.info()

print("\nMissing values:")
print(train_df.isnull().sum())

print("\nDuplicate rows:")
print(train_df.duplicated().sum())

print("\nTarget distribution:")
print(train_df["Revenue"].value_counts())

print("\nTarget percentage:")
print(train_df["Revenue"].value_counts(normalize=True) * 100)

Training data information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9864 entries, 0 to 9863
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Session_ID               9864 non-null   int64  
 1   Administrative           9864 non-null   int64  
 2   Administrative_Duration  9372 non-null   float64
 3   Informational            9864 non-null   int64  
 4   Informational_Duration   9864 non-null   float64
 5   ProductRelated           9864 non-null   int64  
 6   ProductRelated_Duration  9864 non-null   float64
 7   BounceRates              9864 non-null   float64
 8   ExitRates                9173 non-null   float64
 9   PageValues               9864 non-null   float64
 10  SpecialDay               9864 non-null   float64
 11  Month                    9864 non-null   object 
 12  OperatingSystems         9864 non-null   int64  
 13  Browser                  9864 non-null   int64  
 1

In [74]:
print("Missing values in TRAIN:")
print(train_df.isnull().sum())

print("\n" + "="*50 + "\n")

print("Missing values in TEST:")
print(test_df.isnull().sum())

Missing values in TRAIN:
Session_ID                   0
Administrative               0
Administrative_Duration    492
Informational                0
Informational_Duration       0
ProductRelated               0
ProductRelated_Duration      0
BounceRates                  0
ExitRates                  691
PageValues                   0
SpecialDay                   0
Month                        0
OperatingSystems             0
Browser                      0
Region                     591
TrafficType                789
VisitorType                394
Weekend                      0
Revenue                      0
dtype: int64


Missing values in TEST:
Session_ID                   0
Administrative               0
Administrative_Duration    124
Informational                0
Informational_Duration       0
ProductRelated               0
ProductRelated_Duration      0
BounceRates                  0
ExitRates                  172
PageValues                   0
SpecialDay                   0
Month 

In [75]:
target_column = "Revenue"
id_column = "Session_ID"

X = train_df.drop(columns=[target_column, id_column])
y = train_df[target_column]

X_test = test_df.drop(columns=[id_column])

print("Training features:", X.shape)
print("Training target:", y.shape)
print("Test features:", X_test.shape)

Training features: (9864, 17)
Training target: (9864,)
Test features: (2466, 17)


In [76]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nValidation target distribution:")
print(y_val.value_counts(normalize=True))

X_train: (7891, 17)
X_val: (1973, 17)

Training target distribution:
Revenue
False    0.832467
True     0.167533
Name: proportion, dtype: float64

Validation target distribution:
Revenue
False    0.832235
True     0.167765
Name: proportion, dtype: float64


In [77]:
numerical_columns = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_columns = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numerical columns:")
print(numerical_columns)

print("\nCategorical columns:")
print(categorical_columns)

Numerical columns:
['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'OperatingSystems', 'Browser', 'Region', 'TrafficType']

Categorical columns:
['Month', 'VisitorType', 'Weekend']


In [78]:
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_columns),
    ("cat", categorical_pipeline, categorical_columns)
])

In [79]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

In [80]:
model.fit(X_train, y_train)

print("Logistic Regression model trained successfully.")

Logistic Regression model trained successfully.


In [81]:
val_predictions = model.predict(X_val)

validation_accuracy = accuracy_score(
    y_val,
    val_predictions
)

print("Validation Accuracy:", validation_accuracy)

print("\nClassification Report:")
print(
    classification_report(
        y_val,
        val_predictions
    )
)

Validation Accuracy: 0.8672072985301571

Classification Report:
              precision    recall  f1-score   support

       False       0.87      0.98      0.92      1642
        True       0.77      0.30      0.43       331

    accuracy                           0.87      1973
   macro avg       0.82      0.64      0.68      1973
weighted avg       0.86      0.87      0.84      1973



In [82]:
X_processed_array = model.named_steps["preprocessor"].transform(X)

print("Processed feature matrix created.")
print("Processed shape:", X_processed_array.shape)

Processed feature matrix created.
Processed shape: (9864, 29)


In [83]:
feature_names = model.named_steps[
    "preprocessor"
].get_feature_names_out()

print("Number of processed features:", len(feature_names))

Number of processed features: 29


In [84]:
if hasattr(X_processed_array, "toarray"):
    X_processed_array = X_processed_array.toarray()

processed_features = pd.DataFrame(
    X_processed_array,
    columns=feature_names,
    index=train_df.index
)

In [85]:
processed_df = processed_features.copy()

processed_df["Revenue"] = y.values

print("Processed dataframe shape:", processed_df.shape)
print("Missing values in processed_df:")
print(processed_df.isnull().sum().sum())

Processed dataframe shape: (9864, 30)
Missing values in processed_df:
0


In [86]:
original_rows = original_df.shape[0]
processed_rows = processed_df.shape[0]

row_retained_percent = (
    processed_rows / original_rows
) * 100

print("Original rows:", original_rows)
print("Processed rows:", processed_rows)
print("Rows retained:", round(row_retained_percent, 2), "%")

Original rows: 9864
Processed rows: 9864
Rows retained: 100.0 %


In [87]:
model.fit(X, y)

print("Final Logistic Regression model trained on all training data.")

Final Logistic Regression model trained on all training data.


In [88]:
X_test = test_df.drop(columns=["Session_ID"])

print("Test features shape:", X_test.shape)

Test features shape: (2466, 17)


In [89]:
predictions = model.predict(X_test)

print("Number of predictions:", len(predictions))
print("Number of test observations:", len(test_df))

Number of predictions: 2466
Number of test observations: 2466


In [90]:
assert len(predictions) == len(test_df)

print("Every test observation has a prediction.")

Every test observation has a prediction.


In [91]:
print("Prediction distribution:")
print(pd.Series(predictions).value_counts())

print("\nFirst 10 predictions:")
print(predictions[:10])

Prediction distribution:
False    2284
True      182
Name: count, dtype: int64

First 10 predictions:
[False False  True False False False False False False False]


In [92]:
required_variables = [
    "original_df",
    "processed_df",
    "model",
    "predictions",
    "test_df"
]

for variable in required_variables:
    print(
        variable,
        "exists:",
        variable in globals()
    )

original_df exists: True
processed_df exists: True
model exists: True
predictions exists: True
test_df exists: True


In [93]:
print("Original rows:", original_df.shape[0])
print("Processed rows:", processed_df.shape[0])

print(
    "Processed missing values:",
    processed_df.isnull().sum().sum()
)

print(
    "Row retention:",
    round(
        (processed_df.shape[0] / original_df.shape[0]) * 100,
        2
    ),
    "%"
)

print("Test observations:", len(test_df))
print("Predictions:", len(predictions))

Original rows: 9864
Processed rows: 9864
Processed missing values: 0
Row retention: 100.0 %
Test observations: 2466
Predictions: 2466


In [94]:
original_missing = original_df.isnull().sum().sum()
processed_missing = processed_df.isnull().sum().sum()

print("Checkpoint variables created successfully.")
print("Original missing:", original_missing)
print("Processed missing:", processed_missing)
print("Row retention:", row_retained_percent)

Checkpoint variables created successfully.
Original missing: 2957
Processed missing: 0
Row retention: 100.0


In [95]:
original_rows, original_columns = original_df.shape
processed_rows, processed_columns = processed_df.shape

print("Original rows:", original_rows)
print("Original columns:", original_columns)
print("Processed rows:", processed_rows)
print("Processed columns:", processed_columns)

Original rows: 9864
Original columns: 19
Processed rows: 9864
Processed columns: 30


In [96]:
final_model = model

if hasattr(final_model, "best_estimator_"):
    final_model = final_model.best_estimator_

if hasattr(final_model, "steps"):
    final_model = final_model.steps[-1][1]

model_name = final_model.__class__.__name__

print("Model:", model_name)

Model: LogisticRegression


In [97]:
checkpoints = pd.DataFrame({

    "id": [
        "original_missing",
        "processed_missing",
        "original_rows",
        "processed_rows",
        "original_columns",
        "processed_columns",
        "row_retained_percent",
        "model_name"
    ],

    "value": [
        original_missing,
        processed_missing,
        original_rows,
        processed_rows,
        original_columns,
        processed_columns,
        round(row_retained_percent, 2),
        model_name
    ]
})

In [98]:
prediction_output = pd.DataFrame({

    "id": test_df["Session_ID"].astype(str),

    "value": np.asarray(predictions).astype(str)

})

In [99]:
submission = pd.concat(
    [checkpoints, prediction_output],
    ignore_index=True
)

submission.to_csv(
    "submission.csv",
    index=False
)

print("submission.csv created successfully.")
print(checkpoints)

submission.csv created successfully.
                     id               value
0      original_missing                2957
1     processed_missing                   0
2         original_rows                9864
3        processed_rows                9864
4      original_columns                  19
5     processed_columns                  30
6  row_retained_percent               100.0
7            model_name  LogisticRegression


In [100]:
print(submission.head(15))
print("\nTotal rows in submission:", len(submission))
print("Test rows:", len(test_df))

                      id               value
0       original_missing                2957
1      processed_missing                   0
2          original_rows                9864
3         processed_rows                9864
4       original_columns                  19
5      processed_columns                  30
6   row_retained_percent               100.0
7             model_name  LogisticRegression
8                 106094               False
9                 111845               False
10                106794                True
11                103444               False
12                106833               False
13                102684               False
14                110590               False

Total rows in submission: 2474
Test rows: 2466
